# Лабораторная работа 2: NumPy + Pandas + Sklearn
## Анализ цен на жилье в Бостоне
**Студент:** Шибитов  
**Группа:** 11  
**Дата:** 2026-03-11

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans
from scipy import stats

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

## Загрузка данных

Датасет Boston Housing загружается напрямую с GitHub (так как `sklearn.datasets.load_boston` удалён в sklearn >= 1.2).

In [ ]:
from sklearn.datasets import fetch_openml

boston = fetch_openml(name='boston', version=1, as_frame=True, parser='auto')
df_raw = boston.frame
df_raw.head()

In [ ]:
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE',
                'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']

df_raw.columns = column_names
df = df_raw.astype(float).copy()

print('Датасет загружен:', df.shape)
df.head()

---
## Задание 1: Первичный анализ

In [ ]:
# 1.1 Размерность и типы данных
print('Размерность датасета:', df.shape)
print('\nТипы данных:')
print(df.dtypes)

In [ ]:
# 1.2 Проверка пропущенных значений
print('Пропущенные значения по столбцам:')
print(df.isnull().sum())
print(f'\nИтого пропущенных значений: {df.isnull().sum().sum()}')

In [ ]:
# 1.3 Распределение MEDV — гистограмма + boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['MEDV'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Гистограмма MEDV (медианная стоимость жилья)')
axes[0].set_xlabel('MEDV ($1000)')
axes[0].set_ylabel('Частота')

axes[1].boxplot(df['MEDV'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot MEDV')
axes[1].set_ylabel('MEDV ($1000)')

plt.tight_layout()
plt.show()

print(f'Среднее: {df["MEDV"].mean():.2f}  |  Медиана: {df["MEDV"].median():.2f}  |  Std: {df["MEDV"].std():.2f}')

In [ ]:
# 1.4 Признаки с наибольшей корреляцией с MEDV
corr_with_medv = df.corr()['MEDV'].drop('MEDV').sort_values(key=abs, ascending=False)
print('Корреляция признаков с MEDV (по модулю):')
print(corr_with_medv.to_string())

corr_with_medv.plot(kind='bar', color=['green' if x > 0 else 'red' for x in corr_with_medv],
                    figsize=(12, 5), title='Корреляция признаков с MEDV')
plt.ylabel('Корреляция Пирсона')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

**Вывод:** Наибольшая положительная корреляция с MEDV — у признака **RM** (среднее количество комнат): чем больше комнат, тем дороже жильё. Наибольшая отрицательная — у **LSTAT** (доля бедного населения) и **PTRATIO** (соотношение учеников и учителей).

---
## Задание 2: Предобработка

In [ ]:
# 2.1 Нормализация числовых признаков с помощью StandardScaler
df_processed = df.copy()
numeric_cols = [c for c in df.columns if c != 'MEDV']

scaler = StandardScaler()
df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

print('Статистика после нормализации (первые 5 признаков):')
df_processed[numeric_cols[:5]].describe().round(3)

In [ ]:
# 2.2 Бинарный признак is_high_value = MEDV > 30
df_processed['is_high_value'] = (df['MEDV'] > 30).astype(int)
print('Распределение is_high_value:')
print(df_processed['is_high_value'].value_counts())
print(f'Доля дорогого жилья: {df_processed["is_high_value"].mean():.1%}')

In [ ]:
# 2.3 Удаление выбросов по Z-score (> 3 стандартных отклонения)
# Применяем к исходным данным df (не к нормализованным), затем будем работать с df_clean
z_scores = np.abs(stats.zscore(df[numeric_cols]))
outlier_mask = (z_scores > 3).any(axis=1)
df_clean = df[~outlier_mask].copy()

print(f'Строк до удаления выбросов: {len(df)}')
print(f'Строк после удаления выбросов: {len(df_clean)}')
print(f'Удалено выбросов: {outlier_mask.sum()}')

**Вывод:** Нормализация приводит все признаки к единой шкале (среднее=0, std=1). Бинарный признак `is_high_value` выделяет ~16% дорогого жилья. После фильтрации по Z-score удалены выбросы, которые могли бы исказить модель.

---
## Задание 3: Визуализация

In [ ]:
# 3.1 Тепловая карта корреляций
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 8})
plt.title('Тепловая карта корреляций признаков Boston Housing', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Scatter plot: RM vs MEDV
plt.figure(figsize=(9, 6))
plt.scatter(df['RM'], df['MEDV'], alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.3)

# Линия тренда
z = np.polyfit(df['RM'], df['MEDV'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['RM'].min(), df['RM'].max(), 100)
plt.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Тренд: y={z[0]:.2f}x+{z[1]:.2f}')

plt.xlabel('RM (среднее количество комнат)')
plt.ylabel('MEDV (медианная стоимость, $1000)')
plt.title('Зависимость стоимости жилья от количества комнат (RM vs MEDV)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 График зависимости MEDV от LSTAT
plt.figure(figsize=(9, 6))
plt.scatter(df['LSTAT'], df['MEDV'], alpha=0.5, color='darkorange', edgecolors='white', linewidth=0.3)

# Линия тренда
z2 = np.polyfit(df['LSTAT'], df['MEDV'], 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(df['LSTAT'].min(), df['LSTAT'].max(), 100)
plt.plot(x_line2, p2(x_line2), 'b-', linewidth=2, label=f'Тренд: y={z2[0]:.2f}x+{z2[1]:.2f}')

plt.xlabel('LSTAT (% бедного населения)')
plt.ylabel('MEDV (медианная стоимость, $1000)')
plt.title('Зависимость стоимости жилья от доли бедного населения (LSTAT vs MEDV)')
plt.legend()
plt.tight_layout()
plt.show()

**Вывод:** Тепловая карта подтверждает сильные корреляции: RM↑ → MEDV↑, LSTAT↑ → MEDV↓. Scatter plot RM vs MEDV показывает выраженную положительную зависимость. График LSTAT vs MEDV — нелинейную отрицательную зависимость (возможно, логарифмическую).

---
## Задание 4: Линейная регрессия (sklearn)

In [ ]:
from sklearn.model_selection import train_test_split

# Признаки и целевая переменная (на очищенных данных)
features = ['RM', 'LSTAT', 'PTRATIO']
X = df_clean[features].values
y = df_clean['MEDV'].values

# Разбивка на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Обучение модели
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

print('Коэффициенты модели (sklearn):')
for feat, coef in zip(features, model_sklearn.coef_):
    print(f'  {feat}: {coef:.4f}')
print(f'  Intercept: {model_sklearn.intercept_:.4f}')

In [ ]:
# Оценка качества модели
y_pred = model_sklearn.predict(X_test)

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R²   = {r2:.4f}')
print(f'MAE  = {mae:.4f} ($1000)')
print(f'RMSE = {rmse:.4f} ($1000)')

# График предсказаний
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения MEDV')
plt.ylabel('Предсказанные значения MEDV')
plt.title('Реальные vs Предсказанные (Linear Regression sklearn)')
plt.tight_layout()
plt.show()

**Интерпретация коэффициентов:**
- **RM** (+): каждая дополнительная комната увеличивает стоимость примерно на значение коэффициента (тыс. $).
- **LSTAT** (−): чем выше доля бедного населения, тем ниже цена.
- **PTRATIO** (−): чем выше нагрузка на учителей (хуже школы), тем дешевле жильё.

---
## Задание 5: Линейная регрессия вручную (NumPy)

In [ ]:
# Формула МНК: β = (X^T X)^{-1} X^T y

# Добавляем столбец единиц для intercept
X_train_np = np.column_stack([np.ones(len(X_train)), X_train])
X_test_np  = np.column_stack([np.ones(len(X_test)),  X_test])

# Вычисляем коэффициенты
beta = np.linalg.inv(X_train_np.T @ X_train_np) @ X_train_np.T @ y_train

print('Коэффициенты модели (NumPy МНК):')
print(f'  Intercept: {beta[0]:.4f}')
for feat, coef in zip(features, beta[1:]):
    print(f'  {feat}: {coef:.4f}')

In [ ]:
# Сравнение коэффициентов
print('Сравнение sklearn vs NumPy МНК:')
print(f'{"Признак":<12} {"sklearn":>12} {"NumPy":>12} {"Разница":>12}')
print('-' * 50)
all_sk   = [model_sklearn.intercept_] + list(model_sklearn.coef_)
all_np   = list(beta)
all_feat = ['Intercept'] + features
for feat, sk, np_val in zip(all_feat, all_sk, all_np):
    print(f'{feat:<12} {sk:>12.6f} {np_val:>12.6f} {abs(sk-np_val):>12.2e}')

# Предсказания вручную
y_pred_np = X_test_np @ beta
r2_np   = r2_score(y_test, y_pred_np)
mae_np  = mean_absolute_error(y_test, y_pred_np)
rmse_np = np.sqrt(mean_squared_error(y_test, y_pred_np))
print(f'\nНумПай — R²={r2_np:.4f}, MAE={mae_np:.4f}, RMSE={rmse_np:.4f}')

**Вывод:** Коэффициенты, вычисленные вручную через формулу МНК (`β = (XᵀX)⁻¹Xᵀy`), совпадают с результатами sklearn с точностью до числовых ошибок плавающей точки (разница порядка 10⁻¹²). Это подтверждает корректность реализации.

---
## Задание 6: Кластеризация KMeans (дополнительно)

In [ ]:
# Подбор оптимального числа кластеров — метод локтя
cluster_features = ['CRIM', 'NOX', 'DIS']
X_clust = df_clean[cluster_features].values

# Нормализация для кластеризации
scaler_c = StandardScaler()
X_clust_scaled = scaler_c.fit_transform(X_clust)

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), inertias, 'bo-')
plt.xlabel('Число кластеров k')
plt.ylabel('Инерция (WCSS)')
plt.title('Метод локтя для KMeans')
plt.axvline(x=3, color='r', linestyle='--', label='k=3 (выбрано)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# KMeans с k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_clust_scaled)
df_clust = df_clean.copy()
df_clust['Cluster'] = labels

# Визуализация кластеров (CRIM vs DIS)
colors = ['steelblue', 'darkorange', 'green']
plt.figure(figsize=(10, 6))
for i in range(3):
    mask = labels == i
    plt.scatter(df_clust.loc[mask, 'CRIM'], df_clust.loc[mask, 'DIS'],
                alpha=0.6, color=colors[i], label=f'Кластер {i}', edgecolors='white', linewidth=0.3)

plt.xlabel('CRIM (уровень преступности)')
plt.ylabel('DIS (расстояние до деловых центров)')
plt.title('Кластеризация районов по CRIM, NOX, DIS (KMeans k=3)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Средняя цена жилья в каждом кластере
cluster_stats = df_clust.groupby('Cluster').agg(
    MEDV_mean=('MEDV', 'mean'),
    CRIM_mean=('CRIM', 'mean'),
    NOX_mean=('NOX', 'mean'),
    DIS_mean=('DIS', 'mean'),
    Count=('MEDV', 'count')
).round(2)

print('Характеристики кластеров:')
display(cluster_stats)

plt.figure(figsize=(7, 5))
bars = plt.bar(cluster_stats.index, cluster_stats['MEDV_mean'],
               color=colors, edgecolor='grey', alpha=0.85)
for bar, val in zip(bars, cluster_stats['MEDV_mean']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'${val:.1f}k', ha='center', va='bottom', fontweight='bold')
plt.xlabel('Кластер')
plt.ylabel('Средняя MEDV ($1000)')
plt.title('Средняя стоимость жилья по кластерам')
plt.xticks([0, 1, 2], ['Кластер 0', 'Кластер 1', 'Кластер 2'])
plt.tight_layout()
plt.show()

**Вывод по кластеризации:** KMeans выделяет три типа районов:
- Кластер с **высокой преступностью и загрязнением** — самое дешёвое жильё.
- Кластер с **умеренными показателями** — средняя цена.
- Кластер **«спальные районы»** (далеко от центра, чистый воздух, низкая преступность) — наибольшая стоимость.

Это логично: качество среды обитания напрямую влияет на цену жилья.

---
## Общие выводы

1. **Корреляционный анализ** показал, что наиболее значимые факторы цены — RM (+), LSTAT (−) и PTRATIO (−).
2. **Линейная регрессия** на трёх признаках объясняет значительную долю дисперсии (R²≈0.65–0.70). Коэффициенты интерпретируемы.
3. **NumPy МНК** дал идентичные коэффициенты, подтверждая теоретическую корректность.
4. **KMeans** кластеризация выявила осмысленные группы районов с разным уровнем криминальности, экологии и цен.

## Возможные улучшения

- **Полиномиальные признаки** (`PolynomialFeatures`): зависимость LSTAT–MEDV явно нелинейна — квадратичные признаки могут улучшить R².
- **Ridge/Lasso регрессия**: при использовании всех 13 признаков регуляризация предотвратит мультиколлинеарность.
- **Random Forest / Gradient Boosting**: нелинейные ансамблевые методы дают R²≈0.85–0.92 на этом датасете (на кросс-валидации).
- **Логарифмирование LSTAT**: `log(LSTAT)` линеаризует его зависимость с MEDV и улучшает качество линейной модели.